In [33]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [44]:
PDF_PATH = "example.pdf"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "gemma4:31b-cloud"
CHROMA_DIR = "./chroma_db"      
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOP_K = 4

In [45]:
docs = PyPDFLoader("example.pdf").load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
)
chunks = splitter.split_documents(docs)
print(f"Loaded {len(docs)} pages, split into {len(chunks)} chunks.")


Loaded 26 pages, split into 74 chunks.


In [46]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [47]:
llm = ChatOllama(model=LLM_MODEL, temperature=0)

prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer the question using only the context below. "
    "If the context does not contain the answer, say you don't know.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [48]:

while True:
    question = input("\nAsk a question (or 'quit'): ").strip()
    if question.lower() in {"quit", "exit", "q"}:
        break
    print("\nAnswer:", chain.invoke(question))


Answer: Hello! How can I help you?

Answer: The provided text is from ISO/IEC 27001:2022(E) and discusses the requirements for creating, updating, and controlling documented information within an information security management system.

Answer: The provided text states that certain information must be documented and made available to the personnel who need it.

Answer: The provided text contains a table of contents regarding the "Context of the organization" and an "information security management system," as well as specific security controls related to:
* A formalized disciplinary process for information security policy violations.
* Responsibilities and duties that remain valid after termination or change of employment.
* The identification, documentation, and signing of confidentiality or non-disclosure agreements.
* The implementation of security measures for remote working.

Answer: The provided context is a table of contents and does not contain explanations for the listed item